# Markov & Chebyshev inequalities

_Probability & Statistics_

**Bound the chance of extremes using only the mean and variance. No full distribution needed.**

Sometimes you don't know the full distribution — just its mean, maybe its variance.

     These inequalities still let you bound how often extreme values happen.

---

This notebook is a practice scaffold for the **Markov & Chebyshev inequalities** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy + SciPy

Markov and Chebyshev give tail bounds using only the mean (and, for Chebyshev, the variance) — no full distribution required. We test them on an Exponential with mean 1 in three steps: (1) the Markov bound, (2) the Chebyshev bound, (3) a simulation check of how loose Chebyshev is.

### Step 1 — The Markov bound

Markov's inequality says that for a **nonnegative** random variable, `P(X >= a) <= mu / a`. The Exponential(mean 1) is nonnegative, so it qualifies. We compute the bound at `a = 4` and compare it to the true tail probability `P(X >= 4)`, which SciPy gives via the survival function `sf`. The bound should sit above the true tail.

In [ ]:
from scipy.stats import expon

# Exponential with mean 1 (rate 1). Nonnegative, so Markov applies.
mu = 1.0
var = 1.0
d = expon(scale=1.0)

a = 4.0  # Markov: P(X >= a) <= mu / a
markov = mu / a
true_tail = float(d.sf(a))

print("Markov bound  P(X>=4) <=", round(markov, 4))
print("true tail     P(X>=4)  =", round(true_tail, 4))

### Step 2 — The Chebyshev bound

Chebyshev's inequality uses the variance too: `P(|X - mu| >= k·sigma) <= 1/k^2`. It bounds how far a variable strays from its mean in units of standard deviations. Here `sigma = 1`, so at `k = 3` the bound is `1/9`. Notice this needs no assumption about the shape of the distribution.

In [ ]:
k = 3.0  # Chebyshev: P(|X-mu| >= k*sigma) <= 1/k^2
cheby = 1 / k ** 2

print("Chebyshev bound P(|X-mu|>=3sigma) <=", round(cheby, 4))

### Step 3 — Check Chebyshev by simulation

To see how loose the bound is, we draw two million Exponential samples and measure how often a sample actually falls at least `3·sigma` away from the mean. The simulated frequency should be far **below** the `1/9 ≈ 0.111` bound — Chebyshev is guaranteed to hold but is usually pessimistic.

In [ ]:
rng = np.random.default_rng(0)
sigma = var ** 0.5

x = rng.exponential(1.0, size=2_000_000)
far_from_mean = np.abs(x - mu) >= k * sigma

print("sim P(|X-mu|>=3sigma) =", round(np.mean(far_from_mean), 4))

## Visualize the data & results

_How loose are the Markov and Chebyshev tail bounds?_

### Step 1 — Compute the bounds and the true tails

We line up each bound against the actual tail probability it is supposed to cover. For Markov that's `P(X >= 4)`. For Chebyshev we use the upper side `P(X >= mu + 3·sigma)` since an Exponential can't go below 0, leaving the lower side empty. Both true tails come from SciPy's survival function.

In [ ]:
from scipy.stats import expon

# Exponential(mean 1): Markov and Chebyshev bounds vs the true tails.
mu = 1.0
d = expon(scale=1.0)

markov = mu / 4.0  # P(X >= 4) <= mu / a
true_tail = float(d.sf(4.0))

cheby = 1 / 3.0 ** 2  # P(|X-mu| >= 3 sigma) <= 1/k^2
true_cheby = float(d.sf(mu + 3.0))  # sigma = 1, lower side empty

### Step 2 — Plot bounds against true tails

Each orange bar is a guaranteed upper bound; the green bar beside it is the probability it actually covers. The gap between them shows how loose these inequalities are — a price you pay for needing only the mean and variance instead of the full distribution.

In [ ]:
vals = [markov, true_tail, cheby, true_cheby]
labels = ['Markov bound', 'true P(X>=4)', 'Chebyshev bound', 'true P(|X-mu|>=3s)']
colors = ['#ffb454', '#7ee787', '#ffb454', '#7ee787']

plt.bar(labels, vals, color=colors)
plt.title('Tail bounds vs the true tail for Exponential(mean 1)')
plt.ylabel('probability')
plt.show()